In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import numpy as np

from jaxcmr.summarize import (
    calculate_aic_weights,
    generate_t_p_matrices,
    summarize_parameters,
    winner_comparison_matrix,
    raw_winner_comparison_matrix,
    pairwise_aic_differences,
    pairwise_median_aic_differences,
    bayesian_model_selection,
    floor_nested_fitness,
)
from jaxcmr.helpers import find_project_root


def _load_model(model_name, model_title, fit_tag, data_name, fit_dir):
    fit_path = os.path.join(
        find_project_root(), fit_dir,
        f"{data_name}_{model_name}_{fit_tag}.json",
    )
    with open(fit_path) as f:
        result = json.load(f)
    if "subject" not in result["fits"]:
        result["fits"]["subject"] = result["subject"]
    if "allow_repeated_recalls" not in result["fixed"]:
        result["fixed"]["allow_repeated_recalls"] = False
        result["fits"]["allow_repeated_recalls"] = [
            False
        ] * len(result["fits"]["subject"])
    result["name"] = model_title
    if "mfc_trace_sensitivity" in result["free"]:
        result["free"]["repetition_orthogonality"] = result["free"][
            "mfc_trace_sensitivity"
        ]
        result["fits"]["repetition_orthogonality"] = result["fits"][
            "mfc_trace_sensitivity"
        ]
        result["free"].pop("mfc_trace_sensitivity")
        result["fits"].pop("mfc_trace_sensitivity")
    return result


def run_comparison(
    models, run_tag, data_name, fit_dir, target_directory,
    query_parameters, nesting_pairs=None,
):
    results = [
        _load_model(name, title, tag, data_name, fit_dir)
        for name, title, tag in models
    ]
    print(f"Loaded {len(results)} models\n")

    if nesting_pairs:
        n_floored = floor_nested_fitness(results, nesting_pairs)
        print(f"Floored {n_floored} subject-model entries (nesting correction)\n")

    summary = summarize_parameters(
        results, query_parameters, include_std=True, include_ci=True
    )
    table_dir = os.path.join(
        find_project_root(), target_directory, "results", "tables",
    )
    with open(os.path.join(table_dir, f"{data_name}_{run_tag}_parameters.md"), "w") as f:
        f.write(summary)
    print("Parameter Summary:\n")
    print(summary)

    df_t, df_p = generate_t_p_matrices(results)
    print("\nT-Test P-Value Matrix:\n")
    print(df_p.to_markdown())
    print()

    aic_weights = calculate_aic_weights(results)
    with open(os.path.join(table_dir, f"{data_name}_{run_tag}_aic_weights.md"), "w") as f:
        f.write(aic_weights.to_markdown())
    print("\nAIC Weights:\n")
    print(aic_weights.to_markdown())
    print()

    df_comparison = winner_comparison_matrix(results)
    with open(os.path.join(table_dir, f"{data_name}_{run_tag}_winner_ratios.md"), "w") as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))
    print("\nWinner Ratios (AIC):\n")
    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()

    df_comparison = raw_winner_comparison_matrix(results)
    with open(os.path.join(table_dir, f"{data_name}_{run_tag}_raw_winner_ratios.md"), "w") as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))
    print("\nRaw Winner Ratios:\n")
    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()

    mean_delta, ci_halfwidth, _ = pairwise_aic_differences(results)
    delta_aic_table = mean_delta.copy().astype(object)
    for row_label in delta_aic_table.index:
        for col_label in delta_aic_table.columns:
            if row_label == col_label:
                delta_aic_table.loc[row_label, col_label] = ""
                continue
            mean_value = mean_delta.loc[row_label, col_label]
            ci = ci_halfwidth.loc[row_label, col_label]
            if mean_value != mean_value or ci != ci:
                delta_aic_table.loc[row_label, col_label] = ""
            else:
                lower = mean_value - ci
                upper = mean_value + ci
                delta_aic_table.loc[row_label, col_label] = (
                    f"{mean_value:.2f} [{lower:.2f}, {upper:.2f}]"
                )

    with open(os.path.join(table_dir, f"{data_name}_{run_tag}_delta_aic.md"), "w") as f:
        f.write(delta_aic_table.to_markdown())
    print("\nPairwise \u0394AIC (row \u2212 column; mean [CI]):\n")
    print(delta_aic_table.to_markdown())
    print()

    median_delta, median_ci = pairwise_median_aic_differences(results)
    median_table = median_delta.copy().astype(object)
    for row_label in median_table.index:
        for col_label in median_table.columns:
            if row_label == col_label:
                median_table.loc[row_label, col_label] = ""
                continue
            med = median_delta.loc[row_label, col_label]
            ci = median_ci.loc[row_label, col_label]
            if med != med or ci != ci:
                median_table.loc[row_label, col_label] = ""
            else:
                lower = med - ci
                upper = med + ci
                median_table.loc[row_label, col_label] = (
                    f"{med:.2f} [{lower:.2f}, {upper:.2f}]"
                )

    with open(os.path.join(table_dir, f"{data_name}_{run_tag}_median_delta_aic.md"), "w") as f:
        f.write(median_table.to_markdown())
    print("\nPairwise \u0394AIC (row \u2212 column; median [bootstrap CI]):\n")
    print(median_table.to_markdown())
    print()

    bms = bayesian_model_selection(results)
    with open(os.path.join(table_dir, f"{data_name}_{run_tag}_bms.md"), "w") as f:
        f.write(bms.to_markdown(index=False))
    print("\nBayesian Model Selection:\n")
    print(bms.to_markdown(index=False))
    print()

In [3]:
data_name = "TalmiEEG"
fit_dir = "projects/TalmiEEG/results/fits/"
target_directory = "projects/TalmiEEG/"

_old_tag = "50_set_likelihood_fixed_term_best_of_3"
_ecmr_tag = "20260309_50_set_likelihood_best_of_3"

query_parameters = [
    "encoding_drift_rate",
    "start_drift_rate",
    "recall_drift_rate",
    "shared_support",
    "item_support",
    "learning_rate",
    "primacy_scale",
    "primacy_decay",
    "choice_sensitivity",
    "emotion_scale",
    "lpp_main_scale",
    "lpp_main_threshold",
    "lpp_inter_scale",
    "lpp_inter_threshold",
]

# Manuscript Comparison

In [4]:
manuscript_models = [
    ("WeirdCMRNoStop", "CMR", _old_tag),
    ("EEGLPPParsimonious", "CMR+LPP", _old_tag),
    ("eCMREmotionOnly", "eCMR", _ecmr_tag),
    ("eCMREmotionTimesLPP", "LPP-eCMR", _ecmr_tag),
]

manuscript_nesting = [
    ("CMR", "CMR+LPP"),
    ("eCMR", "LPP-eCMR"),
]

run_comparison(
    manuscript_models, "Manuscript_Comparison", data_name, fit_dir,
    target_directory, query_parameters, nesting_pairs=manuscript_nesting,
)

Loaded 4 models

Floored 23 subject-model entries (nesting correction)

Parameter Summary:

| Parameter | Statistic | CMR | CMR+LPP | eCMR | LPP-eCMR |
|---|---|---|---|---|---|
| fitness | mean | 215.01 +/- 15.63 | 213.84 +/- 15.73 | 211.98 +/- 16.31 | 210.83 +/- 16.46 |
|  | std | 46.91 | 47.24 | 48.98 | 49.42 |
|  | min | 130.91 | 129.51 | 114.22 | 113.86 |
|  | max | 318.62 | 316.51 | 315.97 | 315.97 |
| encoding drift rate | mean | 0.66 +/- 0.11 | 0.69 +/- 0.10 | 0.65 +/- 0.10 | 0.73 +/- 0.09 |
|  | std | 0.32 | 0.31 | 0.30 | 0.27 |
|  | min | 0.13 | 0.00 | 0.06 | 0.00 |
|  | max | 1.00 | 1.00 | 1.00 | 1.00 |
| start drift rate | mean | 0.36 +/- 0.12 | 0.42 +/- 0.13 | 0.42 +/- 0.13 | 0.44 +/- 0.13 |
|  | std | 0.35 | 0.38 | 0.39 | 0.39 |
|  | min | 0.00 | 0.00 | 0.00 | 0.00 |
|  | max | 1.00 | 1.00 | 1.00 | 1.00 |
| recall drift rate | mean | 0.57 +/- 0.12 | 0.57 +/- 0.12 | 0.64 +/- 0.10 | 0.64 +/- 0.10 |
|  | std | 0.35 | 0.36 | 0.29 | 0.30 |
|  | min | 0.00 | 0.00 | 0.00 | 0.01 

# Main Comparison

In [5]:
main_models = [
    ("WeirdCMRNoStop", "CMR", _old_tag),
    ("eCMREmotionOnly", "eCMR Emotion", _ecmr_tag),
    ("eCMREmotionTimesLPP", "eCMR Emo\u00d7LPP", _ecmr_tag),
    ("eCMRMainEffects", "eCMR Emo+LPP", _ecmr_tag),
    ("eCMRInteraction", "eCMR Emo+LPP+Int", _ecmr_tag),
    ("eCMREmotionBroad", "eCMR Emotion Broad", _ecmr_tag),
    ("eCMREmotionTimesLPPBroad", "eCMR Emo\u00d7LPP Broad", _ecmr_tag),
]

ecmr_nesting = [
    ("eCMR Emotion", "eCMR Emo\u00d7LPP"),
    ("eCMR Emotion", "eCMR Emo+LPP"),
    ("eCMR Emo+LPP", "eCMR Emo+LPP+Int"),
    ("eCMR Emotion Broad", "eCMR Emo\u00d7LPP Broad"),
]

run_comparison(
    main_models, "eCMR_Main", data_name, fit_dir, target_directory,
    query_parameters, nesting_pairs=ecmr_nesting,
)

Loaded 7 models

Floored 45 subject-model entries (nesting correction)

Parameter Summary:

| Parameter | Statistic | CMR | eCMR Emotion | eCMR Emo×LPP | eCMR Emo+LPP | eCMR Emo+LPP+Int | eCMR Emotion Broad | eCMR Emo×LPP Broad |
|---|---|---|---|---|---|---|---|---|
| fitness | mean | 215.01 +/- 15.63 | 211.98 +/- 16.31 | 210.83 +/- 16.46 | 210.33 +/- 16.31 | 209.62 +/- 16.23 | 211.97 +/- 16.36 | 210.78 +/- 16.52 |
|  | std | 46.91 | 48.98 | 49.42 | 48.95 | 48.74 | 49.12 | 49.61 |
|  | min | 130.91 | 114.22 | 113.86 | 114.22 | 112.84 | 116.18 | 114.44 |
|  | max | 318.62 | 315.97 | 315.97 | 315.52 | 313.45 | 318.57 | 318.22 |
| encoding drift rate | mean | 0.66 +/- 0.11 | 0.65 +/- 0.10 | 0.73 +/- 0.09 | 0.66 +/- 0.10 | 0.75 +/- 0.08 | 0.66 +/- 0.11 | 0.70 +/- 0.10 |
|  | std | 0.32 | 0.30 | 0.27 | 0.30 | 0.24 | 0.32 | 0.30 |
|  | min | 0.13 | 0.06 | 0.00 | 0.06 | 0.28 | 0.00 | 0.00 |
|  | max | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| start drift rate | mean | 0.36 +/- 0.12 

# Full Comparison

In [6]:
full_models = [
    ("WeirdCMRNoStop", "CMR", _old_tag),
    ("EEGEmotionOnly", "Emotion", _old_tag),
    ("EEGLPPOnly", "LPP", _old_tag),
    ("EEGLPPParsimonious", "LPP (k=10)", _old_tag),
    ("EEGLPPExponentOnly", "LPP^exp", _old_tag),
    ("EEGMainEffects", "Emo+LPP", _old_tag),
    ("EEGMainEffectsPlusInteraction", "Emo+LPP+Int", _old_tag),
    ("EEGTwoLayerMainEffects", "2L Emo+LPP", _old_tag),
    ("EEGTwoLayerInteraction", "2L Emo+LPP+Int", _old_tag),
    ("eCMREmotionOnly", "eCMR Emotion", _ecmr_tag),
    ("eCMREmotionTimesLPP", "eCMR Emo\u00d7LPP", _ecmr_tag),
    ("eCMRMainEffects", "eCMR Emo+LPP", _ecmr_tag),
    ("eCMRInteraction", "eCMR Emo+LPP+Int", _ecmr_tag),
    ("eCMREmotionBroad", "eCMR Emotion Broad", _ecmr_tag),
    ("eCMREmotionTimesLPPBroad", "eCMR Emo\u00d7LPP Broad", _ecmr_tag),
]

full_nesting = [
    # Single-context family
    ("CMR", "LPP (k=10)"),
    ("Emotion", "Emo+LPP"),
    ("LPP", "Emo+LPP"),
    ("LPP (k=10)", "Emo+LPP"),
    ("Emo+LPP", "Emo+LPP+Int"),
    # Two-layer family
    ("2L Emo+LPP", "2L Emo+LPP+Int"),
    # eCMR family
    ("eCMR Emotion", "eCMR Emo\u00d7LPP"),
    ("eCMR Emotion", "eCMR Emo+LPP"),
    ("eCMR Emo+LPP", "eCMR Emo+LPP+Int"),
    ("eCMR Emotion Broad", "eCMR Emo\u00d7LPP Broad"),
]

run_comparison(
    full_models, "Full_Comparison", data_name, fit_dir, target_directory,
    query_parameters, nesting_pairs=full_nesting,
)

Loaded 15 models

Floored 106 subject-model entries (nesting correction)

Parameter Summary:

| Parameter | Statistic | CMR | Emotion | LPP | LPP (k=10) | LPP^exp | Emo+LPP | Emo+LPP+Int | 2L Emo+LPP | 2L Emo+LPP+Int | eCMR Emotion | eCMR Emo×LPP | eCMR Emo+LPP | eCMR Emo+LPP+Int | eCMR Emotion Broad | eCMR Emo×LPP Broad |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| fitness | mean | 215.01 +/- 15.63 | 211.74 +/- 16.38 | 213.07 +/- 15.71 | 213.84 +/- 15.73 | 213.28 +/- 15.89 | 210.11 +/- 16.32 | 209.30 +/- 16.23 | 211.77 +/- 16.33 | 211.25 +/- 16.25 | 211.98 +/- 16.31 | 210.83 +/- 16.46 | 210.33 +/- 16.31 | 209.62 +/- 16.23 | 211.97 +/- 16.36 | 210.78 +/- 16.52 |
|  | std | 46.91 | 49.17 | 47.18 | 47.24 | 47.70 | 48.98 | 48.72 | 49.01 | 48.78 | 48.98 | 49.42 | 48.95 | 48.74 | 49.12 | 49.61 |
|  | min | 130.91 | 113.81 | 128.54 | 129.51 | 128.18 | 113.40 | 113.40 | 116.37 | 116.36 | 114.22 | 113.86 | 114.22 | 112.84 | 116.18 | 114.44 |
|  | max | 318.62 | 318.